In [1]:
# Functions to scrape data from wikipedia
import requests
from requests.adapters import HTTPAdapter
from urllib3 import Retry
import random

class ScrapingException(BaseException):
    """Errors gotten when trying to scrape from wikipedia"""

# Use different user agents to mimic different browsers
USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
]

def get_session():
    """
    Function to return a session that I can use to make requests to wikipedia's pages
    """
    session = requests.Session()
    
    # Add retry logic
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter=adapter)
    return session
    
SESSION = get_session()

def get_wikipedia_page(url):
    """
    A function to try getting a wikipedia page
    """
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    # Make a request to Wikipedia
    try:
        print("Getting response")
        wikipedia_response = SESSION.get(url, headers=headers, timeout=15)
        if wikipedia_response.status_code != 200:
            raise ScrapingException(f"Wikipedia refused to send page with status code {wikipedia_response.status_code}")
        else:
            return wikipedia_response
    except Exception as e:
        print(f"Could not get page because of {e}")
        raise Exception("Unknown error")
    
jean_paul_response = get_wikipedia_page("https://en.wikipedia.org/wiki/Jean-Paul_Abalo")

Getting response


In [2]:
from bs4 import BeautifulSoup
import re

class PlayerDetails:
    def __init__(self, from_year: int | None, to_year: int | None, appearances: int | None, goals: int | None, team: str | None):
        self.from_year = from_year
        self.to_year = to_year
        self.appearances = appearances
        self.goals = goals
        self.team = team
        
    def __repr__(self):
        return f"Found data row with from: {self.from_year} to: {self.to_year} team: {self.team} appearances: {self.appearances} and goals: {self.goals}"

def extract_infobox(player_wikipedia: str, problematic_urls: list[str]) -> list[PlayerDetails] | None:
    """
    Extract player details from a player's wikipedia page
    
    Args:
        player_wikipedia: URL of a player's wikipedia page
        problematic_urls: list of wikipedia pages that couldn't be parsed
        
    Returns:
        players: A list of player objects if response of player was parsable
    """
    player_response = get_wikipedia_page(player_wikipedia)
    assert player_response.status_code == 200, "Can't parse an unsucessful response"
    
    # Parse HTML of the player's Wiki page
    soup = BeautifulSoup(player_response.text, 'html.parser')

    # Look for the players info box
    tables = soup.select_one("table.infobox")

    # Store unparsable player for later processing
    if not tables:
        print(f"Player does not have info box in wiki: {player_wikipedia}")
        problematic_urls.append(player_wikipedia)
        return None

    player_details = []
    table_rows = tables.find_all("tr")
    current = 0
    section_name = ""
    content_num = 0

    # Scanning table rows
    while True:
        if current >= len(table_rows):
            print("Consumed all rows in table")
            break
        
        current_row = table_rows[current]
        
        # Checking if row is the heading of a section
        if len(current_row.find_all("td")) == 0:
            section_name = current_row.select_one("th").text
            content_num = 0
            
            # Check if it's the section we want
            if "Senior career" in section_name:
                # While next item is content
                while current + 1 < len(table_rows) and len(table_rows[current + 1].find_all("td")) > 0:
                    next_content = table_rows[current + 1]
                    # Check if it is a table row
                    if len(next_content.find_all("b")) > 0:
                        print(f"Found title row {next_content.find("b")}")
                    else:
                        # Work with data that looks like 1 th 3 td
                        data_timeline = next_content.find("th")
                        
                        from_year = None
                        to_year = None
                        team = None
                        appearances = None
                        goals = None
                        
                        # Extract from and to years
                        if data_timeline:
                            raw_timeline = data_timeline.get_text(strip=True)
                            if len(raw_timeline) > 4:
                                from_year = int(re.sub(r'\D', '', raw_timeline[0:4]))
                                to_year = int(re.sub(r'\D', '', raw_timeline[4:]))
                            else:
                                from_year = int(re.sub(r'\D', '', raw_timeline))
                        
                        rest_data = next_content.find_all("td")
                        if len(rest_data) < 3:
                            print(f"Unkown rest data {rest_data}")
                        # Get team, appearance, goals
                        else:
                            team = rest_data[0].get_text(strip=True)
                            if len(rest_data[1].get_text(strip=True)) > 0:
                                appearances = int(re.sub(r'\D', '', rest_data[1].get_text(strip=True)))
                            if len(rest_data[2].get_text(strip=True)) > 0:
                                goals = int(re.sub(r'\D', '', rest_data[2].get_text(strip=True)))
                    
                        print(f"Found data row with from: {from_year} to: {to_year} team: {team} appearances: {appearances} and goals: {goals}")
                        player_details.append(PlayerDetails(
                            from_year=from_year,
                            to_year=to_year,
                            appearances=appearances,
                            goals=goals,
                            team = team
                        ))
                        content_num += 1
                    
                    current += 1
                    
                # Increment
                current += 1
                print(f"Section {section_name} had {content_num} items")
            else:
                current += 1
        else:
            current += 1
            
    return player_details
        
problematic_players = []
player_info = extract_infobox("https://en.wikipedia.org/wiki/Jean-Paul_Abalo", problematic_players)
print(player_info)

Getting response
Found title row <b>Team</b>
Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None
Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appearances: 29 and goals: 1
Found data row with from: 1995 to: 2005 team: Amiens SC appearances: 273 and goals: 7
Found data row with from: 2005 to: None team: USL Dunkerque appearances: 4 and goals: 0
Found data row with from: 2006 to: None team: APOEL appearances: 3 and goals: 0
Found data row with from: 2006 to: None team: Ethnikos Piraeus appearances: 9 and goals: 0
Found data row with from: 2007 to: 2008 team: Al-Merrikh appearances: None and goals: None
Found data row with from: 2008 to: 2009 team: FC Déols 36 appearances: None and goals: None
Section Senior career* had 8 items
Consumed all rows in table
[Found data row with from: 1992 to: 1993 team: OC Agaza appearances: None and goals: None, Found data row with from: 1993 to: 1995 team: Saint-Christophe Châteauroux appear

In [11]:
class Player:
    def __init__(self, id: str, wikipedia_url: str, problematic_urls: list[str]):
        self.id = id
        self.wikipedia_url = wikipedia_url
        self.wikipedia_details = self._extract_wikipedia_details(problematic_urls=problematic_urls)
        
    def _extract_wikipedia_details(self, problematic_urls: list[str]) -> list[PlayerDetails] | None:
        """
        Returns player details that have been extracted from their wikipedia page
        
        Args:
            problematic_urls : A list used to store all urls that had an issue extracting details from
            
        Returns:
            player_details: A list of player details from wikipedia
        """
        return extract_infobox(self.wikipedia_url, problematic_urls=problematic_urls)
    
    def to_csv(self) -> list[str]:
        """ 
        Returns strings that will represent the player in a CSV file
        
        Returns:
            entries (str): Entries in CSV file
        """
        entries = []
        for detail in self.wikipedia_details:
            entries.append(f"{self.id},{detail.appearances},{detail.team},{detail.goals},{detail.from_year},{detail.to_year}\n")
            
        return entries
    
    def __repr__(self):
        return f"Player: {self.wikipedia_details}"

In [13]:
raw_players = [
    {'id': 'P001', 'url': 'https://en.wikipedia.org/wiki/Alan_A%27Court'},
    {'id': 'P001', 'url': 'https://en.wikipedia.org/wiki/Stefan_Abadzhiev'},
    {'id': 'P001', 'url': 'https://en.wikipedia.org/wiki/Patrice_Abanda'}
]

players = []

for raw in raw_players:
    players.append(
        Player(raw['id'], wikipedia_url=raw['url'], problematic_urls=problematic_players)
    )
    
print(players)

Getting response
Found title row <b>Team</b>
Found data row with from: 1952 to: 1964 team: Liverpool appearances: 354 and goals: 61
Found data row with from: 1964 to: 1966 team: Tranmere Rovers appearances: 50 and goals: 11
Found data row with from: 1966 to: 1967 team: Norwich City appearances: 0 and goals: 0
Found title row <b>404</b>
Section Senior career* had 3 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1953 to: 1968 team: Levski Sofia appearances: 254 and goals: 37
Found data row with from: 1968 to: 1970 team: Wiesbaden appearances: None and goals: None
Section Senior career* had 2 items
Consumed all rows in table
Getting response
Found title row <b>Team</b>
Found data row with from: 1995 to: 1998 team: Tonnerre Yaoundé appearances: 0 and goals: 0
Found data row with from: 1998 to: 1999 team: PAOK appearances: 0 and goals: 0
Found data row with from: 1999 to: 2000 team: Apollon Kalamarias appearances: 0 and goals: 0
Found

In [14]:
from datetime import datetime
def export_players_csv(players: list[Player]):
    """
    Function that will store each of the player details in a newly generated CSV file
    """
    file_name = f"players_{datetime.now()}.csv"
    
    with open(file_name, 'a') as file:
        # Insert csv file header
        file.write("id,appearances,team,goals,from,to\n")
        
        # Insert entries in CSV
        for player in players:
            entries = player.to_csv()
            for entry in entries:
                file.write(entry)
    
export_players_csv(players)